<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/ROSG_Test_H1_StochasticActivation_DeltaTau_WindowSweep_ZetaPoles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG_Test_H1_StochasticActivation_DeltaTau_WindowSweep_ZetaPoles

Self-contained Colab-ready audit. Tests absolute windows and windows relative to each activation center Zc^(i).

In [1]:
# ============================================
# ROSG_Test_H1_StochasticActivation_DeltaTau_WindowSweep_ZetaPoles
# Continuity with ROSG_Test_H1_StochasticActivation_ZetaPoles_Audit
#
# Purpose:
#   Test whether stochastic H1 activation may exhibit DSI-like pseudo-poles only
#   in an intermediate DeltaTau/Z window, rather than on the whole Z-domain.
#
# Central question:
#   Is there a mesoscopic operational-resolution window Z=ln(DeltaTau_sun/t_P)
#   where active/non-active switching chi_i(Z) preserves a specific pseudo-pole
#   / log-periodic signature, while the UV noisy and IR smoothed regimes do not?
#
# Discipline:
#   - H1 branch receives no log-periodic injection.
#   - Positive branch injects log-periodicity for detector validation only.
#   - Window-local detrending is used.
#   - A candidate window requires H1 coherence, null-control rejection, power margin,
#     and a viable activation regime: neither near-absent nor saturated.
#   - Finite graphs/processes only support pseudo-poles/resonance peaks, not rigorous
#     meromorphic zeta poles.
# ============================================

import json
import math
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


@dataclass
class WindowSweepConfig:
    levels: Tuple[int, ...] = (1, 2, 3, 4, 5)
    tP_s: float = 5.391246446661944e-44
    Z_min: float = 0.0
    Z_max: float = 6.5
    n_Z: int = 91
    Zc0: float = 1.05
    delta_Z_level: float = math.log(2.0)
    activation_width: float = 0.34
    n_realizations: int = 16
    markov_refresh_rate: float = 0.18
    min_activation_prob: float = 0.02
    max_activation_prob: float = 0.92
    stationary_control_p: float = 0.45
    g_min: float = 0.0
    g_max: float = 0.35
    u_probe: float = 1.0
    # Window sweep: widths and step in Z.
    window_widths_Z: Tuple[float, ...] = (1.5, 2.0)
    window_step_Z: float = 1.0
    min_points_per_window: int = 14
    # Local period scan.
    period_min_Z: float = 0.38
    period_max_Z_global: float = 2.80
    n_periods_Z: int = 20
    min_cycles_for_candidate: float = 1.50
    n_permutations: int = 4
    fap_threshold: float = 0.25
    min_power_threshold: float = 0.10
    coherence_cv_threshold: float = 0.25
    min_levels_for_coherent_claim: int = 3
    power_margin_threshold: float = 1.25
    # Viability of stochastic window.
    activation_mean_low: float = 0.20
    activation_mean_high: float = 0.80
    activation_std_low: float = 0.05
    activation_std_high: float = 0.35
    # Activation zeta pseudo-poles on active-run lengths.
    D_activation_zeta: float = 2.0
    omega_min_zeta: float = 0.5
    omega_max_zeta: float = 24.0
    n_omega_zeta: int = 24
    n_zeta_null_bootstrap: int = 4
    zeta_zscore_threshold: float = 2.5
    min_active_runs: int = 10
    # Positive control only.
    positive_control_amplitude: float = 0.28
    positive_control_period_Z: float = math.log(2.0)
    positive_control_phase0: float = 0.2
    # Detrending.
    trend_poly_degree: int = 4
    out_dir: str = "/mnt/data/ROSG_Test_H1_StochasticActivation_DeltaTau_WindowSweep_ZetaPoles_out"
    random_seed: int = 20260708


def level_center(cfg: WindowSweepConfig, i: int, nonlattice: bool = False) -> float:
    if nonlattice:
        return cfg.Zc0 + (i - 1) * cfg.delta_Z_level * math.sqrt(2.0) + 0.05 * math.sin(1.17 * i)
    return cfg.Zc0 + (i - 1) * cfg.delta_Z_level


def activation_probability(Z: np.ndarray, cfg: WindowSweepConfig, i: int, case: str) -> np.ndarray:
    if case == "CTRL_H0_no_channel":
        return np.zeros_like(Z)
    if case == "CTRL_stationary_markov_no_Z":
        return np.full_like(Z, cfg.stationary_control_p)
    nonlattice = case == "CTRL_Markov_nonlattice_centers"
    zc = level_center(cfg, i, nonlattice=nonlattice)
    base = cfg.min_activation_prob + (cfg.max_activation_prob - cfg.min_activation_prob) * 0.5 * (
        1.0 + np.tanh((Z - zc) / max(cfg.activation_width, 1e-12))
    )
    if case == "POS_injected_periodic_activation_detector_check":
        omega0 = 2.0 * math.pi / cfg.positive_control_period_Z
        phase = cfg.positive_control_phase0 + 0.37 * (i - 1)
        base = base + cfg.positive_control_amplitude * np.cos(omega0 * Z + phase)
    return np.clip(base, 0.001, 0.999)


def simulate_chi_markov(p: np.ndarray, cfg: WindowSweepConfig, rng: np.random.Generator, independent: bool = False) -> np.ndarray:
    nR, nZ = cfg.n_realizations, len(p)
    if independent:
        return (rng.random((nR, nZ)) < p[None, :]).astype(np.int8)
    chi = np.zeros((nR, nZ), dtype=np.int8)
    chi[:, 0] = (rng.random(nR) < p[0]).astype(np.int8)
    for t in range(1, nZ):
        refresh = rng.random(nR) < cfg.markov_refresh_rate
        new_state = (rng.random(nR) < p[t]).astype(np.int8)
        chi[:, t] = np.where(refresh, new_state, chi[:, t - 1])
    return chi


def simulate_case_level(cfg: WindowSweepConfig, case: str, i: int, Z: np.ndarray, rng: np.random.Generator):
    if case == "CTRL_Bernoulli_independent_lattice":
        p = activation_probability(Z, cfg, i, "H1_stochastic_Z_coupled")
        chi = simulate_chi_markov(p, cfg, rng, independent=True)
    else:
        p_case = "H1_stochastic_Z_coupled" if case == "CTRL_Z_shuffled_H1_sequence" else case
        p = activation_probability(Z, cfg, i, p_case)
        chi = simulate_chi_markov(p, cfg, rng, independent=False)
        if case == "CTRL_Z_shuffled_H1_sequence":
            for r in range(chi.shape[0]):
                chi[r] = rng.permutation(chi[r])
            p = np.full_like(p, p.mean())
    return p, chi


def detrend_local(x: np.ndarray, y: np.ndarray, cfg: WindowSweepConfig):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if len(x) < 4:
        trend = np.full_like(y, np.mean(y))
        return trend, y - trend
    degree = min(cfg.trend_poly_degree, max(1, len(x) - 2))
    xs = 2.0 * (x - x.min()) / max(x.max() - x.min(), 1e-12) - 1.0
    coef = np.polyfit(xs, y, deg=degree)
    trend = np.polyval(coef, xs)
    return trend, y - trend


def precompute_basis(x: np.ndarray, periods: np.ndarray) -> np.ndarray:
    omegas = 2.0 * np.pi / periods
    Qs = np.zeros((len(omegas), len(x), 2), dtype=float)
    for k, w in enumerate(omegas):
        X = np.column_stack([np.cos(w * x), np.sin(w * x)])
        X = X - X.mean(axis=0, keepdims=True)
        Q, _ = np.linalg.qr(X)
        Qs[k] = Q[:, :2]
    return Qs


def powers_for_residual_with_basis(r: np.ndarray, Qs: np.ndarray) -> np.ndarray:
    r = np.asarray(r, float)
    r = r - r.mean()
    ss = float(np.sum(r * r))
    if ss <= 1e-20:
        return np.zeros(Qs.shape[0])
    proj = np.einsum("knd,n->kd", Qs, r)
    return np.clip(np.sum(proj * proj, axis=1) / ss, 0.0, 1.0)


def fit_phase_amp(x: np.ndarray, r: np.ndarray, period: float):
    w = 2.0 * math.pi / period
    X = np.column_stack([np.cos(w * x), np.sin(w * x), np.ones_like(x)])
    beta, *_ = np.linalg.lstsq(X, r, rcond=None)
    a, b, c = [float(v) for v in beta]
    return float(math.hypot(a, b)), float(math.atan2(-b, a)), float(c)


def scan_Z_periodicity(x: np.ndarray, residual: np.ndarray, cfg: WindowSweepConfig, rng: np.random.Generator, width_Z: float):
    period_max = min(cfg.period_max_Z_global, max(cfg.period_min_Z * 1.05, width_Z * 0.80))
    periods = np.linspace(cfg.period_min_Z, period_max, cfg.n_periods_Z)
    Qs = precompute_basis(x, periods)
    r = np.asarray(residual, float)
    sd = float(np.std(r))
    if sd <= 1e-14 or len(x) < cfg.min_points_per_window:
        return {"best_period_Z": float("nan"), "best_omega_Z": float("nan"), "best_power_Z": 0.0,
                "best_amp_std_Z": 0.0, "best_phase_Z": float("nan"), "fap_Z": 1.0,
                "n_cycles_best": 0.0, "period_scan_min_Z": float(cfg.period_min_Z), "period_scan_max_Z": float(period_max)}
    r_std = (r - r.mean()) / sd
    powers = powers_for_residual_with_basis(r_std, Qs)
    j = int(np.argmax(powers))
    period = float(periods[j])
    amp, phase, offset = fit_phase_amp(x, r_std, period)
    null_max = np.zeros(cfg.n_permutations, dtype=float)
    for k in range(cfg.n_permutations):
        null_max[k] = float(np.max(powers_for_residual_with_basis(rng.permutation(r_std), Qs)))
    fap = (1.0 + float(np.sum(null_max >= powers[j]))) / (cfg.n_permutations + 1.0)
    return {
        "best_period_Z": period,
        "best_omega_Z": float(2.0 * math.pi / period),
        "best_power_Z": float(powers[j]),
        "best_amp_std_Z": float(amp),
        "best_phase_Z": float(phase),
        "best_offset_std_Z": float(offset),
        "fap_Z": float(fap),
        "n_cycles_best": float(width_Z / max(period, 1e-12)),
        "null_max_power_Z_median": float(np.median(null_max)),
        "null_max_power_Z_q95": float(np.quantile(null_max, 0.95)),
        "period_scan_min_Z": float(cfg.period_min_Z),
        "period_scan_max_Z": float(period_max),
    }


def active_run_lengths_from_chi_window(chi: np.ndarray, dZ: float) -> np.ndarray:
    lengths = []
    for row in chi:
        run = 0
        for val in row:
            if val == 1:
                run += 1
            else:
                if run > 0:
                    lengths.append(run * dZ)
                    run = 0
        if run > 0:
            lengths.append(run * dZ)
    return np.asarray(lengths, dtype=float)


def zeta_activation_scan(lengths: np.ndarray, cfg: WindowSweepConfig, rng: np.random.Generator):
    L = np.asarray(lengths, float)
    L = L[np.isfinite(L) & (L > 0)]
    if len(L) < cfg.min_active_runs:
        return {"n_active_runs": int(len(L)), "zeta_best_omega": float("nan"), "zeta_best_period_log_length": float("nan"),
                "zeta_best_power": 0.0, "zeta_null_mean": 0.0, "zeta_null_std": 0.0, "zeta_zscore": 0.0,
                "zeta_fap_bootstrap": 1.0, "zeta_pseudopole_pass": False,
                "pseudo_pole_real_D": float(cfg.D_activation_zeta), "pseudo_pole_imag_omega": float("nan")}
    omega = np.linspace(cfg.omega_min_zeta, cfg.omega_max_zeta, cfg.n_omega_zeta)
    logL = np.log(L)
    weights = L ** cfg.D_activation_zeta
    norm = max(float(np.sum(weights)), 1e-30)
    resp = np.abs(np.exp(1j * np.outer(omega, logL)) @ weights) / norm
    power = resp ** 2
    j = int(np.argmax(power))
    null_max = np.zeros(cfg.n_zeta_null_bootstrap, dtype=float)
    meanL = float(np.mean(L))
    for b in range(cfg.n_zeta_null_bootstrap):
        Lb = rng.exponential(scale=max(meanL, 1e-6), size=len(L))
        Lb = np.clip(Lb, np.min(L), np.max(L) * 1.2)
        wb = Lb ** cfg.D_activation_zeta
        rb = np.abs(np.exp(1j * np.outer(omega, np.log(Lb))) @ wb) / max(float(np.sum(wb)), 1e-30)
        null_max[b] = float(np.max(rb ** 2))
    zscore = (float(power[j]) - float(np.mean(null_max))) / max(float(np.std(null_max)), 1e-12)
    fap = (1.0 + float(np.sum(null_max >= power[j]))) / (len(null_max) + 1.0)
    return {
        "n_active_runs": int(len(L)),
        "active_run_mean_Z": float(np.mean(L)),
        "active_run_median_Z": float(np.median(L)),
        "zeta_best_omega": float(omega[j]),
        "zeta_best_period_log_length": float(2.0 * math.pi / omega[j]),
        "zeta_best_power": float(power[j]),
        "zeta_null_mean": float(np.mean(null_max)),
        "zeta_null_std": float(np.std(null_max)),
        "zeta_zscore": float(zscore),
        "zeta_fap_bootstrap": float(fap),
        "zeta_pseudopole_pass": bool(zscore >= cfg.zeta_zscore_threshold and fap <= cfg.fap_threshold),
        "pseudo_pole_real_D": float(cfg.D_activation_zeta),
        "pseudo_pole_imag_omega": float(omega[j]),
    }


def spectral_response_curve(cfg: WindowSweepConfig, i: int, Z: np.ndarray, p_hat: np.ndarray) -> np.ndarray:
    zc = level_center(cfg, i, nonlattice=False)
    g = cfg.g_min + (cfg.g_max - cfg.g_min) * 0.5 * (1.0 + np.tanh((Z - zc) / max(cfg.activation_width, 1e-12)))
    response = 0.5 * (1.0 - np.exp(-2.0 * cfg.u_probe * g))
    return p_hat * response


def build_windows(cfg: WindowSweepConfig) -> List[Tuple[float, float, float]]:
    windows = []
    for width in cfg.window_widths_Z:
        starts = np.arange(cfg.Z_min, cfg.Z_max - width + 1e-9, cfg.window_step_Z)
        for a in starts:
            windows.append((float(a), float(a + width), float(width)))
    return windows


def coherence(values: np.ndarray, min_count: int, threshold: float):
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if len(v) < min_count:
        return False, float("inf")
    cv = float(np.std(v) / max(abs(np.mean(v)), 1e-12))
    return bool(cv <= threshold), cv


def generate_global_curves(cfg: WindowSweepConfig, rng: np.random.Generator):
    Z = np.linspace(cfg.Z_min, cfg.Z_max, cfg.n_Z)
    DeltaTau_s = cfg.tP_s * np.exp(Z)
    dZ = float(Z[1] - Z[0])
    cases = [
        "H1_stochastic_Z_coupled",
        "CTRL_H0_no_channel",
        "CTRL_Bernoulli_independent_lattice",
        "CTRL_stationary_markov_no_Z",
        "CTRL_Markov_nonlattice_centers",
        "CTRL_Z_shuffled_H1_sequence",
        "POS_injected_periodic_activation_detector_check",
    ]
    rows = []
    chi_store = {}
    for case in cases:
        for i in cfg.levels:
            p, chi = simulate_case_level(cfg, case, i, Z, rng)
            p_hat = chi.mean(axis=0)
            spec = spectral_response_curve(cfg, i, Z, p_hat) if case != "CTRL_H0_no_channel" else np.zeros_like(Z)
            chi_store[(case, i)] = chi
            for idx, (zz, dt, pp, ph, sc) in enumerate(zip(Z, DeltaTau_s, p, p_hat, spec)):
                rows.append({
                    "case": case,
                    "target_i": int(i),
                    "Z_index": int(idx),
                    "Z": float(zz),
                    "DeltaTau_s": float(dt),
                    "p_theoretical": float(pp),
                    "p_activation_empirical": float(ph),
                    "spectral_heat_contrast_u_probe": float(sc),
                })
    return pd.DataFrame(rows), chi_store, Z, DeltaTau_s, dZ, cases


def audit_window_case_level(cfg: WindowSweepConfig, case: str, i: int, a: float, b: float, width: float, global_curves: pd.DataFrame, chi_store: dict, rng: np.random.Generator, dZ: float):
    d = global_curves[(global_curves["case"] == case) & (global_curves["target_i"] == i) & (global_curves["Z"] >= a - 1e-12) & (global_curves["Z"] <= b + 1e-12)].copy()
    if len(d) < cfg.min_points_per_window:
        return None
    x = d["Z"].to_numpy(float)
    y = d["p_activation_empirical"].to_numpy(float)
    spec = d["spectral_heat_contrast_u_probe"].to_numpy(float)
    trend, resid = detrend_local(x, y, cfg)
    zscan = scan_Z_periodicity(x, resid, cfg, rng, width)
    spec_trend, spec_resid = detrend_local(x, spec, cfg)
    spec_scan = scan_Z_periodicity(x, spec_resid, cfg, rng, width)
    idxs = d["Z_index"].to_numpy(int)
    chi_window = chi_store[(case, i)][:, idxs]
    lengths = active_run_lengths_from_chi_window(chi_window, dZ)
    zeta = zeta_activation_scan(lengths, cfg, rng)
    mean_act = float(np.mean(y))
    std_act = float(np.std(y))
    # Viable means: H1 is neither effectively absent nor fully smoothed/saturated; std not white-noise huge.
    viable = bool(cfg.activation_mean_low <= mean_act <= cfg.activation_mean_high and cfg.activation_std_low <= std_act <= cfg.activation_std_high)
    activation_pass = bool(zscan["fap_Z"] <= cfg.fap_threshold and zscan["best_power_Z"] >= cfg.min_power_threshold and zscan["n_cycles_best"] >= cfg.min_cycles_for_candidate)
    spectral_pass = bool(spec_scan["fap_Z"] <= cfg.fap_threshold and spec_scan["best_power_Z"] >= cfg.min_power_threshold and spec_scan["n_cycles_best"] >= cfg.min_cycles_for_candidate)
    return {
        "case": case,
        "target_i": int(i),
        "Z_window_start": float(a),
        "Z_window_end": float(b),
        "Z_window_width": float(width),
        "DeltaTau_start_s": float(cfg.tP_s * math.exp(a)),
        "DeltaTau_end_s": float(cfg.tP_s * math.exp(b)),
        "n_points_window": int(len(d)),
        "mean_activation_empirical": mean_act,
        "std_activation_empirical": std_act,
        "activation_window_viable": viable,
        "activation_Z_periodicity_pass": activation_pass,
        **zscan,
        **zeta,
        "spectral_contrast_Z_periodicity_pass": spectral_pass,
        "spectral_best_period_Z": spec_scan["best_period_Z"],
        "spectral_best_omega_Z": spec_scan["best_omega_Z"],
        "spectral_best_power_Z": spec_scan["best_power_Z"],
        "spectral_fap_Z": spec_scan["fap_Z"],
        "spectral_n_cycles_best": spec_scan["n_cycles_best"],
        "expected_lattice_period_ln2": float(math.log(2.0)),
        "abs_best_period_minus_ln2": float(abs(zscan["best_period_Z"] - math.log(2.0))) if np.isfinite(zscan["best_period_Z"]) else float("nan"),
        "pseudo_pole_notation_activation": f"s = {cfg.D_activation_zeta:.6g} + i*{zeta['pseudo_pole_imag_omega']:.6g}",
    }


def run_window_sweep_audit(cfg: WindowSweepConfig):
    out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(cfg.random_seed)
    prefix = "ROSG_Test_H1_StochasticActivation_DeltaTau_WindowSweep_ZetaPoles"

    global_curves, chi_store, Z, DeltaTau_s, dZ, cases = generate_global_curves(cfg, rng)
    windows = build_windows(cfg)

    rows = []
    for a, b, w in windows:
        for case in cases:
            for i in cfg.levels:
                rec = audit_window_case_level(cfg, case, i, a, b, w, global_curves, chi_store, rng, dZ)
                if rec is not None:
                    rows.append(rec)
    level_window = pd.DataFrame(rows)

    # Summarize each case inside each window.
    summary_rows = []
    keycols = ["Z_window_start", "Z_window_end", "Z_window_width"]
    for key, dwin in level_window.groupby(keycols + ["case"]):
        a, b, width, case = key
        d = dwin.copy()
        dviable = d[d["activation_window_viable"]]
        z_pass = d[(d["activation_Z_periodicity_pass"]) & (d["activation_window_viable"])]
        zeta_pass = d[(d["zeta_pseudopole_pass"]) & (d["activation_window_viable"])]
        spec_pass = d[(d["spectral_contrast_Z_periodicity_pass"]) & (d["activation_window_viable"])]
        z_coh, z_cv = coherence(z_pass["best_period_Z"].to_numpy(float), cfg.min_levels_for_coherent_claim, cfg.coherence_cv_threshold)
        zeta_coh, zeta_cv = coherence(zeta_pass["zeta_best_omega"].to_numpy(float), cfg.min_levels_for_coherent_claim, cfg.coherence_cv_threshold)
        spec_coh, spec_cv = coherence(spec_pass["spectral_best_period_Z"].to_numpy(float), cfg.min_levels_for_coherent_claim, cfg.coherence_cv_threshold)
        # We include level-viability as a separate flag; a candidate window needs >=3 viable levels.
        n_viable = int(len(dviable))
        summary_rows.append({
            "case": case,
            "Z_window_start": float(a),
            "Z_window_end": float(b),
            "Z_window_width": float(width),
            "DeltaTau_start_s": float(cfg.tP_s * math.exp(a)),
            "DeltaTau_end_s": float(cfg.tP_s * math.exp(b)),
            "n_levels": int(len(d)),
            "n_viable_levels": n_viable,
            "median_mean_activation": float(d["mean_activation_empirical"].median()),
            "median_std_activation": float(d["std_activation_empirical"].median()),
            "n_activation_Z_periodicity_pass_viable": int(len(z_pass)),
            "activation_Z_coherent_multilevel": bool(len(z_pass) >= cfg.min_levels_for_coherent_claim and z_coh),
            "activation_Z_period_cv": z_cv,
            "median_activation_best_power_Z": float(d["best_power_Z"].median()),
            "median_activation_fap_Z": float(d["fap_Z"].median()),
            "median_activation_best_period_Z": float(d["best_period_Z"].median()),
            "n_zeta_pseudopole_pass_viable": int(len(zeta_pass)),
            "activation_zeta_pseudopoles_coherent": bool(len(zeta_pass) >= cfg.min_levels_for_coherent_claim and zeta_coh),
            "activation_zeta_omega_cv": zeta_cv,
            "median_zeta_best_power": float(d["zeta_best_power"].median()),
            "median_zeta_zscore": float(d["zeta_zscore"].median()),
            "median_zeta_best_omega": float(d["zeta_best_omega"].median()),
            "n_spectral_contrast_Z_periodicity_pass_viable": int(len(spec_pass)),
            "spectral_contrast_Z_coherent_multilevel": bool(len(spec_pass) >= cfg.min_levels_for_coherent_claim and spec_coh),
            "spectral_contrast_period_cv": spec_cv,
            "median_spectral_best_power_Z": float(d["spectral_best_power_Z"].median()),
            "median_spectral_fap_Z": float(d["spectral_fap_Z"].median()),
        })
    window_case_summary = pd.DataFrame(summary_rows)

    # Verdict per window: H1 must pass in viable window, controls rejected, power margin.
    window_verdicts = []
    null_cases = [c for c in cases if c not in ["H1_stochastic_Z_coupled", "POS_injected_periodic_activation_detector_check"]]
    for (a, b, width), wsum in window_case_summary.groupby(keycols):
        h1 = wsum[wsum["case"] == "H1_stochastic_Z_coupled"]
        null = wsum[wsum["case"].isin(null_cases)]
        pos = wsum[wsum["case"] == "POS_injected_periodic_activation_detector_check"]
        if len(h1) == 0:
            continue
        h1_row = h1.iloc[0]
        h1_viable = bool(h1_row["n_viable_levels"] >= cfg.min_levels_for_coherent_claim)
        h1_activation_pass = bool(h1_row["activation_Z_coherent_multilevel"])
        h1_zeta_pass = bool(h1_row["activation_zeta_pseudopoles_coherent"])
        h1_spec_pass = bool(h1_row["spectral_contrast_Z_coherent_multilevel"])
        h1_combined = bool(h1_viable and (h1_activation_pass or h1_spec_pass) and h1_zeta_pass)
        null_coh = sorted(null[(null["activation_Z_coherent_multilevel"] | null["activation_zeta_pseudopoles_coherent"] | null["spectral_contrast_Z_coherent_multilevel"])] ["case"].tolist())
        null_rejected = bool(len(null_coh) == 0)
        h1_power = max(float(h1_row["median_activation_best_power_Z"]), float(h1_row["median_zeta_best_power"]), float(h1_row["median_spectral_best_power_Z"]))
        null_power = max(float(null["median_activation_best_power_Z"].max()) if len(null) else 0.0,
                         float(null["median_zeta_best_power"].max()) if len(null) else 0.0,
                         float(null["median_spectral_best_power_Z"].max()) if len(null) else 0.0)
        margin = float(h1_power / max(null_power, 1e-12))
        margin_pass = bool(margin >= cfg.power_margin_threshold)
        pos_ok = bool(len(pos) and (bool(pos.iloc[0]["activation_Z_coherent_multilevel"]) or bool(pos.iloc[0]["activation_zeta_pseudopoles_coherent"]) or bool(pos.iloc[0]["spectral_contrast_Z_coherent_multilevel"])))
        candidate = bool(h1_combined and null_rejected and margin_pass)
        if candidate:
            verdict = "candidate_window_specific_stochastic_zeta_poles_controls_rejected"
        elif h1_combined and not null_rejected:
            verdict = "window_H1_pseudopoles_not_specific_controls_pass"
        elif h1_viable and (h1_activation_pass or h1_spec_pass) and not h1_zeta_pass:
            verdict = "window_H1_switching_coherent_but_no_zeta_pseudopoles"
        elif not h1_viable:
            verdict = "window_not_viable_activation_absent_or_saturated_or_noise_extreme"
        else:
            verdict = "window_no_H1_DSI_candidate"
        window_verdicts.append({
            "Z_window_start": float(a),
            "Z_window_end": float(b),
            "Z_window_width": float(width),
            "DeltaTau_start_s": float(cfg.tP_s * math.exp(a)),
            "DeltaTau_end_s": float(cfg.tP_s * math.exp(b)),
            "verdict": verdict,
            "candidate_window": candidate,
            "h1_viable_activation_window": h1_viable,
            "h1_activation_Z_coherent_pass": h1_activation_pass,
            "h1_activation_zeta_pseudopoles_coherent_pass": h1_zeta_pass,
            "h1_spectral_contrast_Z_coherent_pass": h1_spec_pass,
            "h1_combined_candidate_pass": h1_combined,
            "null_controls_rejected": null_rejected,
            "null_controls_or_surrogates_coherent": null_coh,
            "h1_power_margin_over_max_null": margin,
            "power_margin_pass": margin_pass,
            "positive_control_detector_ok": pos_ok,
            "h1_median_mean_activation": float(h1_row["median_mean_activation"]),
            "h1_median_std_activation": float(h1_row["median_std_activation"]),
            "h1_median_activation_best_period_Z": float(h1_row["median_activation_best_period_Z"]),
            "h1_median_zeta_best_omega": float(h1_row["median_zeta_best_omega"]),
            "h1_median_zeta_zscore": float(h1_row["median_zeta_zscore"]),
        })
    window_verdicts = pd.DataFrame(window_verdicts)

    # Relative-to-activation-center sweep. This is crucial because the viable transition
    # window drifts with i: Zc^(i)=Zc0+(i-1)ln2. Absolute windows may miss the same
    # dynamical phase across levels. Each level is therefore tested around its own Zc^(i),
    # and then grouped by the same relative interval.
    relative_patterns = [
        ("rel_minus1p0_to_plus0p5", -1.0, 0.5),
        ("rel_minus0p75_to_plus0p75", -0.75, 0.75),
        ("rel_minus0p5_to_plus1p0", -0.5, 1.0),
        ("rel_minus1p0_to_plus1p0", -1.0, 1.0),
    ]
    rel_rows = []
    for rel_label, rel_a, rel_b in relative_patterns:
        for case in cases:
            for i in cfg.levels:
                zc = level_center(cfg, i, nonlattice=False)
                a = max(cfg.Z_min, zc + rel_a)
                b = min(cfg.Z_max, zc + rel_b)
                width = b - a
                if width <= 0:
                    continue
                rec = audit_window_case_level(cfg, case, i, a, b, width, global_curves, chi_store, rng, dZ)
                if rec is not None:
                    rec["relative_window_label"] = rel_label
                    rec["relative_window_start_from_Zc"] = float(rel_a)
                    rec["relative_window_end_from_Zc"] = float(rel_b)
                    rec["window_family"] = "relative_to_Zc_lattice"
                    rel_rows.append(rec)
    relative_level_window = pd.DataFrame(rel_rows)

    rel_summary_rows = []
    if len(relative_level_window):
        for key, dwin in relative_level_window.groupby(["relative_window_label", "case"]):
            rel_label, case = key
            d = dwin.copy()
            dviable = d[d["activation_window_viable"]]
            z_pass = d[(d["activation_Z_periodicity_pass"]) & (d["activation_window_viable"])]
            zeta_pass = d[(d["zeta_pseudopole_pass"]) & (d["activation_window_viable"])]
            spec_pass = d[(d["spectral_contrast_Z_periodicity_pass"]) & (d["activation_window_viable"])]
            z_coh, z_cv = coherence(z_pass["best_period_Z"].to_numpy(float), cfg.min_levels_for_coherent_claim, cfg.coherence_cv_threshold)
            zeta_coh, zeta_cv = coherence(zeta_pass["zeta_best_omega"].to_numpy(float), cfg.min_levels_for_coherent_claim, cfg.coherence_cv_threshold)
            spec_coh, spec_cv = coherence(spec_pass["spectral_best_period_Z"].to_numpy(float), cfg.min_levels_for_coherent_claim, cfg.coherence_cv_threshold)
            rel_summary_rows.append({
                "relative_window_label": rel_label,
                "case": case,
                "relative_window_start_from_Zc": float(d["relative_window_start_from_Zc"].iloc[0]),
                "relative_window_end_from_Zc": float(d["relative_window_end_from_Zc"].iloc[0]),
                "n_levels": int(len(d)),
                "n_viable_levels": int(len(dviable)),
                "median_mean_activation": float(d["mean_activation_empirical"].median()),
                "median_std_activation": float(d["std_activation_empirical"].median()),
                "n_activation_Z_periodicity_pass_viable": int(len(z_pass)),
                "activation_Z_coherent_multilevel": bool(len(z_pass) >= cfg.min_levels_for_coherent_claim and z_coh),
                "activation_Z_period_cv": z_cv,
                "median_activation_best_power_Z": float(d["best_power_Z"].median()),
                "median_activation_fap_Z": float(d["fap_Z"].median()),
                "median_activation_best_period_Z": float(d["best_period_Z"].median()),
                "n_zeta_pseudopole_pass_viable": int(len(zeta_pass)),
                "activation_zeta_pseudopoles_coherent": bool(len(zeta_pass) >= cfg.min_levels_for_coherent_claim and zeta_coh),
                "activation_zeta_omega_cv": zeta_cv,
                "median_zeta_best_power": float(d["zeta_best_power"].median()),
                "median_zeta_zscore": float(d["zeta_zscore"].median()),
                "median_zeta_best_omega": float(d["zeta_best_omega"].median()),
                "n_spectral_contrast_Z_periodicity_pass_viable": int(len(spec_pass)),
                "spectral_contrast_Z_coherent_multilevel": bool(len(spec_pass) >= cfg.min_levels_for_coherent_claim and spec_coh),
                "spectral_contrast_period_cv": spec_cv,
                "median_spectral_best_power_Z": float(d["spectral_best_power_Z"].median()),
                "median_spectral_fap_Z": float(d["spectral_fap_Z"].median()),
            })
    relative_case_summary = pd.DataFrame(rel_summary_rows)

    relative_verdict_rows = []
    if len(relative_case_summary):
        for rel_label, rsum in relative_case_summary.groupby("relative_window_label"):
            h1 = rsum[rsum["case"] == "H1_stochastic_Z_coupled"]
            null = rsum[rsum["case"].isin(null_cases)]
            pos = rsum[rsum["case"] == "POS_injected_periodic_activation_detector_check"]
            if len(h1) == 0:
                continue
            h1_row = h1.iloc[0]
            h1_viable = bool(h1_row["n_viable_levels"] >= cfg.min_levels_for_coherent_claim)
            h1_activation_pass = bool(h1_row["activation_Z_coherent_multilevel"])
            h1_zeta_pass = bool(h1_row["activation_zeta_pseudopoles_coherent"])
            h1_spec_pass = bool(h1_row["spectral_contrast_Z_coherent_multilevel"])
            h1_combined = bool(h1_viable and (h1_activation_pass or h1_spec_pass) and h1_zeta_pass)
            null_coh = sorted(null[(null["activation_Z_coherent_multilevel"] | null["activation_zeta_pseudopoles_coherent"] | null["spectral_contrast_Z_coherent_multilevel"])] ["case"].tolist())
            null_rejected = bool(len(null_coh) == 0)
            h1_power = max(float(h1_row["median_activation_best_power_Z"]), float(h1_row["median_zeta_best_power"]), float(h1_row["median_spectral_best_power_Z"]))
            null_power = max(float(null["median_activation_best_power_Z"].max()) if len(null) else 0.0,
                             float(null["median_zeta_best_power"].max()) if len(null) else 0.0,
                             float(null["median_spectral_best_power_Z"].max()) if len(null) else 0.0)
            margin = float(h1_power / max(null_power, 1e-12))
            margin_pass = bool(margin >= cfg.power_margin_threshold)
            pos_ok = bool(len(pos) and (bool(pos.iloc[0]["activation_Z_coherent_multilevel"]) or bool(pos.iloc[0]["activation_zeta_pseudopoles_coherent"]) or bool(pos.iloc[0]["spectral_contrast_Z_coherent_multilevel"])))
            candidate = bool(h1_combined and null_rejected and margin_pass)
            if candidate:
                verdict_rel = "relative_candidate_window_specific_stochastic_zeta_poles_controls_rejected"
            elif h1_combined and not null_rejected:
                verdict_rel = "relative_window_H1_pseudopoles_not_specific_controls_pass"
            elif h1_viable and (h1_activation_pass or h1_spec_pass) and not h1_zeta_pass:
                verdict_rel = "relative_window_H1_switching_coherent_but_no_zeta_pseudopoles"
            elif not h1_viable:
                verdict_rel = "relative_window_not_viable_activation_absent_or_saturated_or_noise_extreme"
            else:
                verdict_rel = "relative_window_no_H1_DSI_candidate"
            relative_verdict_rows.append({
                "relative_window_label": rel_label,
                "relative_window_start_from_Zc": float(h1_row["relative_window_start_from_Zc"]),
                "relative_window_end_from_Zc": float(h1_row["relative_window_end_from_Zc"]),
                "verdict": verdict_rel,
                "candidate_window": candidate,
                "h1_viable_activation_window": h1_viable,
                "h1_activation_Z_coherent_pass": h1_activation_pass,
                "h1_activation_zeta_pseudopoles_coherent_pass": h1_zeta_pass,
                "h1_spectral_contrast_Z_coherent_pass": h1_spec_pass,
                "h1_combined_candidate_pass": h1_combined,
                "null_controls_rejected": null_rejected,
                "null_controls_or_surrogates_coherent": null_coh,
                "h1_power_margin_over_max_null": margin,
                "power_margin_pass": margin_pass,
                "positive_control_detector_ok": pos_ok,
                "h1_median_mean_activation": float(h1_row["median_mean_activation"]),
                "h1_median_std_activation": float(h1_row["median_std_activation"]),
                "h1_median_activation_best_period_Z": float(h1_row["median_activation_best_period_Z"]),
                "h1_median_zeta_best_omega": float(h1_row["median_zeta_best_omega"]),
                "h1_median_zeta_zscore": float(h1_row["median_zeta_zscore"]),
            })
    relative_verdicts = pd.DataFrame(relative_verdict_rows)


    # Overall verdict.
    candidates_abs = window_verdicts[window_verdicts["candidate_window"]] if len(window_verdicts) else pd.DataFrame()
    viable_abs = window_verdicts[window_verdicts["h1_viable_activation_window"]] if len(window_verdicts) else pd.DataFrame()
    candidates_rel = relative_verdicts[relative_verdicts["candidate_window"]] if len(relative_verdicts) else pd.DataFrame()
    viable_rel = relative_verdicts[relative_verdicts["h1_viable_activation_window"]] if len(relative_verdicts) else pd.DataFrame()
    if len(candidates_abs) > 0 or len(candidates_rel) > 0:
        overall_verdict = "H1_DeltaTau_window_specific_stochastic_zeta_poles_candidate_found"
        claim = True
    elif len(viable_abs) > 0 or len(viable_rel) > 0:
        overall_verdict = "H1_DeltaTau_window_sweep_no_specific_zeta_poles_in_viable_windows"
        claim = False
    else:
        overall_verdict = "H1_DeltaTau_window_sweep_no_viable_intermediate_window_detected"
        claim = False

    # Identify best non-claim window by score (absolute + relative).
    best_candidates_for_score = []
    if len(window_verdicts):
        wa = window_verdicts.copy(); wa["window_mode"] = "absolute"
        best_candidates_for_score.append(wa)
    if len(relative_verdicts):
        wr = relative_verdicts.copy(); wr["window_mode"] = "relative_to_Zc"
        best_candidates_for_score.append(wr)
    if best_candidates_for_score:
        allv = pd.concat(best_candidates_for_score, ignore_index=True, sort=False)
        viable_allv = allv[allv["h1_viable_activation_window"].astype(bool)].copy()
        score_df = viable_allv if len(viable_allv) else allv
        score = (score_df["h1_median_zeta_zscore"].fillna(0).to_numpy(float)
                 + score_df["h1_power_margin_over_max_null"].fillna(0).to_numpy(float)
                 + 0.5 * score_df["h1_viable_activation_window"].astype(float).to_numpy())
        best_idx = int(np.argmax(score))
        best_window = score_df.iloc[best_idx].to_dict()
    else:
        best_window = {}

    report = {
        "test_name": prefix,
        "status": "completed",
        "verdict": overall_verdict,
        "levels_tested": list(cfg.levels),
        "Z_definition": "Z = ln(DeltaTau_sun / t_P)",
        "window_sweep": {
            "window_widths_Z": list(cfg.window_widths_Z),
            "window_step_Z": cfg.window_step_Z,
            "n_windows_tested": int(len(window_verdicts)),
            "n_absolute_windows_tested": int(len(window_verdicts)),
            "n_relative_windows_tested": int(len(relative_verdicts)),
            "n_candidate_windows_absolute": int(len(candidates_abs)),
            "n_candidate_windows_relative": int(len(candidates_rel)),
            "n_h1_viable_windows_absolute": int(len(viable_abs)),
            "n_h1_viable_windows_relative": int(len(viable_rel)),
        },
        "candidate_windows_absolute": candidates_abs.to_dict(orient="records") if len(candidates_abs) else [],
        "candidate_windows_relative": candidates_rel.to_dict(orient="records") if len(candidates_rel) else [],
        "best_window_by_internal_score": best_window,
        "stochastic_activation_zeta_pole_window_claim": claim,
        "important_scope_note": "This window sweep tests whether a mesoscopic DeltaTau/Z interval supports H1-specific pseudo-poles. H1 receives stochastic switching but no log-periodic injection; positive control is detector-only. Finite simulations detect pseudo-poles/resonance peaks, not rigorous meromorphic poles.",
        "config": asdict(cfg),
    }

    # Save outputs.
    global_curves.to_csv(out / f"{prefix}_global_curves.csv", index=False)
    level_window.to_csv(out / f"{prefix}_window_level_metrics.csv", index=False)
    window_case_summary.to_csv(out / f"{prefix}_window_case_summary.csv", index=False)
    window_verdicts.to_csv(out / f"{prefix}_window_verdicts.csv", index=False)
    relative_level_window.to_csv(out / f"{prefix}_relative_window_level_metrics.csv", index=False)
    relative_case_summary.to_csv(out / f"{prefix}_relative_window_case_summary.csv", index=False)
    relative_verdicts.to_csv(out / f"{prefix}_relative_window_verdicts.csv", index=False)
    def _json_clean(obj):
        if isinstance(obj, dict):
            return {k: _json_clean(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [_json_clean(v) for v in obj]
        if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
            return None
        return obj
    (out / f"{prefix}_report.json").write_text(json.dumps(_json_clean(report), indent=2, allow_nan=False), encoding="utf-8")
    make_plots(prefix, out, cfg, global_curves, window_verdicts, window_case_summary)
    make_package(prefix, out)
    return report, global_curves, level_window, window_case_summary, window_verdicts


def make_plots(prefix: str, out: Path, cfg: WindowSweepConfig, global_curves: pd.DataFrame, window_verdicts: pd.DataFrame, window_case_summary: pd.DataFrame):
    if plt is None:
        return
    # Plot global activation to show UV/transition/IR regimes.
    fig, ax = plt.subplots(figsize=(9, 5))
    for i in cfg.levels:
        d = global_curves[(global_curves["case"] == "H1_stochastic_Z_coupled") & (global_curves["target_i"] == i)]
        ax.plot(d["Z"], d["p_activation_empirical"], label=f"H1 i={i}")
    ax.axhspan(0.2, 0.8, alpha=0.12, label="viable mean activation band")
    ax.set_title("H1 stochastic activation across global Z")
    ax.set_xlabel("Z = ln(DeltaTau/tP)")
    ax.set_ylabel("E[chi_i(Z)]")
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout(); fig.savefig(out / f"{prefix}_global_H1_activation.png", dpi=160); plt.close(fig)

    # Window verdict flags by center.
    if len(window_verdicts):
        w = window_verdicts.copy()
        w["Z_center"] = 0.5 * (w["Z_window_start"] + w["Z_window_end"])
        fig, ax = plt.subplots(figsize=(10, 5))
        for width, d in w.groupby("Z_window_width"):
            ax.plot(d["Z_center"], d["h1_power_margin_over_max_null"], marker="o", label=f"W={width:g}")
        ax.axhline(cfg.power_margin_threshold, linestyle="--", linewidth=1.0, label="margin threshold")
        ax.set_title("H1 power margin over strongest null by Z window")
        ax.set_xlabel("Z window center")
        ax.set_ylabel("H1/null power margin")
        ax.legend(fontsize=8)
        fig.tight_layout(); fig.savefig(out / f"{prefix}_window_power_margin.png", dpi=160); plt.close(fig)

        fig, ax = plt.subplots(figsize=(10, 5))
        for width, d in w.groupby("Z_window_width"):
            ax.plot(d["Z_center"], d["h1_median_zeta_zscore"], marker="o", label=f"W={width:g}")
        ax.axhline(cfg.zeta_zscore_threshold, linestyle="--", linewidth=1.0, label="zeta zscore threshold")
        ax.set_title("H1 activation-zeta pseudo-pole z-score by Z window")
        ax.set_xlabel("Z window center")
        ax.set_ylabel("median zeta z-score")
        ax.legend(fontsize=8)
        fig.tight_layout(); fig.savefig(out / f"{prefix}_window_zeta_zscore.png", dpi=160); plt.close(fig)

        flag_cols = ["h1_viable_activation_window", "h1_activation_Z_coherent_pass", "h1_activation_zeta_pseudopoles_coherent_pass", "h1_spectral_contrast_Z_coherent_pass", "null_controls_rejected", "candidate_window"]
        fig, ax = plt.subplots(figsize=(11, 5))
        plot_w = w.sort_values(["Z_window_width", "Z_window_start"]).reset_index(drop=True)
        mat = plot_w[flag_cols].astype(int).to_numpy()
        ax.imshow(mat.T, aspect="auto")
        ax.set_yticks(np.arange(len(flag_cols))); ax.set_yticklabels(flag_cols, fontsize=8)
        ax.set_xticks(np.linspace(0, len(plot_w)-1, min(12, len(plot_w))).astype(int))
        labels = [f"{plot_w.iloc[k]['Z_window_start']:.1f}-{plot_w.iloc[k]['Z_window_end']:.1f}\nW{plot_w.iloc[k]['Z_window_width']:.1f}" for k in ax.get_xticks()]
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
        ax.set_title("Window-sweep verdict flags")
        fig.tight_layout(); fig.savefig(out / f"{prefix}_window_verdict_flags.png", dpi=160); plt.close(fig)

        # Best periods for H1 per window.
        h1sum = window_case_summary[window_case_summary["case"] == "H1_stochastic_Z_coupled"].copy()
        h1sum["Z_center"] = 0.5 * (h1sum["Z_window_start"] + h1sum["Z_window_end"])
        fig, ax = plt.subplots(figsize=(10, 5))
        for width, d in h1sum.groupby("Z_window_width"):
            ax.plot(d["Z_center"], d["median_activation_best_period_Z"], marker="o", label=f"W={width:g}")
        ax.axhline(math.log(2.0), linestyle="--", linewidth=1.0, label="ln 2")
        ax.set_title("H1 median best activation period by Z window")
        ax.set_xlabel("Z window center")
        ax.set_ylabel("best period in Z")
        ax.legend(fontsize=8)
        fig.tight_layout(); fig.savefig(out / f"{prefix}_window_best_periods.png", dpi=160); plt.close(fig)


def make_package(prefix: str, out: Path):
    zip_path = Path("/mnt/data") / f"{prefix}_package.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in out.glob(f"{prefix}_*"):
            if p.is_file():
                z.write(p, arcname=f"{out.name}/{p.name}")
        script_path = Path("/mnt/data") / f"{prefix}.py"
        if script_path.exists():
            z.write(script_path, arcname=script_path.name)
        nb_path = Path("/mnt/data") / f"{prefix}.ipynb"
        if nb_path.exists():
            z.write(nb_path, arcname=nb_path.name)
    return zip_path


if __name__ == "__main__":
    cfg = WindowSweepConfig()
    report, global_curves, level_window, window_case_summary, window_verdicts = run_window_sweep_audit(cfg)
    print(json.dumps({
        "status": report["status"],
        "test_name": report["test_name"],
        "verdict": report["verdict"],
        "levels_tested": report["levels_tested"],
        "window_sweep": report["window_sweep"],
        "candidate_windows_absolute": report["candidate_windows_absolute"],
        "candidate_windows_relative": report["candidate_windows_relative"],
        "best_window_by_internal_score": report["best_window_by_internal_score"],
        "stochastic_activation_zeta_pole_window_claim": report["stochastic_activation_zeta_pole_window_claim"],
    }, indent=2))


{
  "status": "completed",
  "test_name": "ROSG_Test_H1_StochasticActivation_DeltaTau_WindowSweep_ZetaPoles",
  "verdict": "H1_DeltaTau_window_sweep_no_specific_zeta_poles_in_viable_windows",
  "levels_tested": [
    1,
    2,
    3,
    4,
    5
  ],
  "window_sweep": {
    "window_widths_Z": [
      1.5,
      2.0
    ],
    "window_step_Z": 1.0,
    "n_windows_tested": 11,
    "n_absolute_windows_tested": 11,
    "n_relative_windows_tested": 4,
    "n_candidate_windows_absolute": 0,
    "n_candidate_windows_relative": 0,
    "n_h1_viable_windows_absolute": 0,
    "n_h1_viable_windows_relative": 4
  },
  "candidate_windows_absolute": [],
  "candidate_windows_relative": [],
  "best_window_by_internal_score": {
    "Z_window_start": NaN,
    "Z_window_end": NaN,
    "Z_window_width": NaN,
    "DeltaTau_start_s": NaN,
    "DeltaTau_end_s": NaN,
    "verdict": "relative_window_H1_pseudopoles_not_specific_controls_pass",
    "candidate_window": false,
    "h1_viable_activation_window": tr

# ROSG_Test_H1_MesoscopicDSI_Discriminant_Audit

Self-contained audit notebook generated from the standalone Python script. Run all cells to reproduce outputs.

In [5]:
%%writefile /mnt/data/ROSG_Test_H1_MesoscopicDSI_Discriminant_Audit.py
import json
import math
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:  # pragma: no cover
    plt = None


@dataclass
class Config:
    test_name: str = "ROSG_Test_H1_MesoscopicDSI_Discriminant_Audit"
    levels: Tuple[int, ...] = (1, 2, 3, 4, 5)
    tP_s: float = 5.391246446661944e-44
    Z_min: float = 0.0
    Z_max: float = 6.5
    n_Z: int = 181
    Zc0: float = 1.05
    delta_Z_level: float = math.log(2.0)
    activation_width: float = 0.34
    min_activation_prob: float = 0.02
    max_activation_prob: float = 0.92
    g_max: float = 0.35
    n_realizations: int = 20
    markov_refresh_rate: float = 0.17
    random_seed: int = 20260708

    # Relative windows zeta = Z-Zc(i)
    relative_windows: Tuple[Tuple[float, float], ...] = (
        (-0.75, 0.75),
        (-0.50, 1.00),
        (-0.25, 1.25),
    )
    min_points_per_window: int = 36

    # Period scan in relative Z.
    period_min_Z: float = 0.38
    period_max_Z: float = 1.60
    n_periods_Z: int = 42
    n_permutations: int = 6

    # Viability criteria.
    activation_mean_low: float = 0.20
    activation_mean_high: float = 0.80
    activation_std_low: float = 0.035
    activation_std_high: float = 0.42
    min_heat_contrast: float = 0.004

    # Specificity criteria.
    min_levels_for_coherent_claim: int = 3
    coherence_cv_threshold: float = 0.18
    detrend_period_drift_threshold: float = 0.10
    omega_consistency_cv_threshold: float = 0.18
    power_margin_weak: float = 1.50
    power_margin_strong: float = 2.00
    fap_weak: float = 0.05
    fap_strong: float = 0.01
    null_pass_power_threshold: float = 0.18

    # Positive control only.
    positive_control_amplitude: float = 0.22
    positive_control_period_Z: float = math.log(2.0)
    positive_control_phase0: float = 0.2

    # Heat proxy.
    n_heat_modes_base: int = 80
    u_probe_grid: Tuple[float, ...] = (0.18, 0.27, 0.40, 0.60, 0.90, 1.35, 2.0)

    out_dir: str = "/mnt/data/ROSG_Test_H1_MesoscopicDSI_Discriminant_Audit_out"


def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def level_center(cfg: Config, i: int, nonlattice: bool = False) -> float:
    if nonlattice:
        # deliberately incommensurate with ln 2
        return cfg.Zc0 + (i - 1) * cfg.delta_Z_level * math.sqrt(2.0) + 0.07 * math.sin(1.11 * i)
    return cfg.Zc0 + (i - 1) * cfg.delta_Z_level


def activation_probability(Z: np.ndarray, cfg: Config, i: int, case: str) -> np.ndarray:
    nonlattice = case in {"CTRL_nonlattice_matched_activation"}
    zc = level_center(cfg, i, nonlattice=nonlattice)
    base = cfg.min_activation_prob + (cfg.max_activation_prob - cfg.min_activation_prob) * sigmoid((Z - zc) / cfg.activation_width)

    if case == "CTRL_markov_stationary_no_Z":
        return np.full_like(Z, float(np.mean(base)))
    if case == "CTRL_matched_heatkernel_envelope":
        # Smooth envelope-only control; used mainly for heat observable.
        return base
    if case == "POS_logperiodic_detector_check":
        zeta = Z - level_center(cfg, i, nonlattice=False)
        band = np.exp(-0.5 * ((zeta - 0.20) / 0.72) ** 2)
        phase = cfg.positive_control_phase0 + 0.18 * (i - 1)
        base = base + cfg.positive_control_amplitude * band * np.cos(2 * np.pi * zeta / cfg.positive_control_period_Z + phase)
    return np.clip(base, 1e-4, 1 - 1e-4)


def simulate_markov_from_p(p: np.ndarray, cfg: Config, rng: np.random.Generator, refresh_rate: float | None = None, independent: bool = False) -> np.ndarray:
    nR, nT = cfg.n_realizations, len(p)
    if independent:
        return (rng.random((nR, nT)) < p[None, :]).astype(np.int8)
    rr = cfg.markov_refresh_rate if refresh_rate is None else refresh_rate
    chi = np.zeros((nR, nT), dtype=np.int8)
    chi[:, 0] = (rng.random(nR) < p[0]).astype(np.int8)
    for t in range(1, nT):
        refresh = rng.random(nR) < rr
        proposed = (rng.random(nR) < p[t]).astype(np.int8)
        chi[:, t] = np.where(refresh, proposed, chi[:, t - 1])
    return chi


def run_lengths(binary: np.ndarray) -> List[int]:
    out: List[int] = []
    for row in binary:
        if len(row) == 0:
            continue
        current = int(row[0])
        run = 1
        for val in row[1:]:
            val = int(val)
            if val == current:
                run += 1
            else:
                if current == 1:
                    out.append(run)
                current = val
                run = 1
        if current == 1:
            out.append(run)
    return out


def reconstruct_from_run_lengths(run_lengths_on: List[int], nR: int, nT: int, p_mean: float, rng: np.random.Generator) -> np.ndarray:
    """Semi-Markov matched dwell: preserve active run lengths, randomize placements."""
    if not run_lengths_on:
        return (rng.random((nR, nT)) < p_mean).astype(np.int8)
    chi = np.zeros((nR, nT), dtype=np.int8)
    for r in range(nR):
        t = 0
        state = int(rng.random() < p_mean)
        while t < nT:
            if state == 1:
                L = int(rng.choice(run_lengths_on))
            else:
                # geometric-like off durations matched roughly to p_mean
                q = max(0.03, min(0.97, p_mean))
                L = int(max(1, rng.geometric(q)))
            end = min(nT, t + L)
            if state == 1:
                chi[r, t:end] = 1
            t = end
            state = 1 - state
        # random circular phase destroys Z placement but keeps durations
        shift = int(rng.integers(0, nT))
        chi[r] = np.roll(chi[r], shift)
    return chi


def shuffled_by_level(chi: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    out = np.empty_like(chi)
    for r in range(chi.shape[0]):
        out[r] = rng.permutation(chi[r])
    return out


def phase_scramble_curve(y: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    y0 = y - np.mean(y)
    spec = np.fft.rfft(y0)
    if len(spec) > 2:
        phases = rng.uniform(0, 2 * np.pi, len(spec) - 2)
        spec[1:-1] *= np.exp(1j * phases)
    return np.fft.irfft(spec, n=len(y)) + np.mean(y)


def planar_spectrum_1d_proxy(i: int, cfg: Config) -> np.ndarray:
    # Compact deterministic proxy for a local Laplacian spectrum. It preserves
    # low/mid band heat-trace sensitivity without building large graphs.
    n = cfg.n_heat_modes_base + 8 * i
    k = np.arange(1, n + 1, dtype=float)
    # 2D grid-like spectral growth with level-dependent scaling.
    scale = 4.0 ** max(i - 1, 0)
    lam = scale * (0.25 + (k / n) ** 1.15 * 8.0)
    return lam


def heat_contrast_from_activation(act: np.ndarray, i: int, cfg: Config) -> np.ndarray:
    lam = planar_spectrum_1d_proxy(i, cfg)
    u = np.array(cfg.u_probe_grid, dtype=float)
    P0 = np.array([np.mean(np.exp(-uu * lam)) for uu in u])
    g = cfg.g_max * np.clip(act, 0, 1)
    out = []
    for gg in g:
        P1 = np.array([0.5 * (np.mean(np.exp(-uu * lam)) + np.mean(np.exp(-uu * (lam + 2.0 * gg)))) for uu in u])
        out.append(np.mean(np.log(np.maximum(P1, 1e-300)) - np.log(np.maximum(P0, 1e-300))))
    return np.array(out)


def spectral_contrast_from_activation(act: np.ndarray, i: int, cfg: Config) -> np.ndarray:
    # A monotone but nonlinear proxy of spectral contrast; it is intentionally
    # distinct from heat_contrast while still driven by H1 activation.
    a = np.clip(act, 0, 1)
    return 0.28 * (1 - np.exp(-3.1 * cfg.g_max * a)) * (1.0 + 0.03 * math.log1p(i))


def moving_average(y: np.ndarray, window: int) -> np.ndarray:
    window = max(3, int(window) | 1)
    if window >= len(y):
        return np.full_like(y, float(np.mean(y)))
    pad = window // 2
    yy = np.pad(y, pad, mode="edge")
    kernel = np.ones(window) / window
    return np.convolve(yy, kernel, mode="valid")


def detrend(x: np.ndarray, y: np.ndarray, method: str) -> Tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if method.startswith("poly"):
        deg = int(method.replace("poly", ""))
        deg = min(deg, max(1, len(x) - 2))
        xs = 2 * (x - x.min()) / max(x.max() - x.min(), 1e-12) - 1
        coef = np.polyfit(xs, y, deg)
        tr = np.polyval(coef, xs)
        return tr, y - tr
    if method == "moving":
        tr = moving_average(y, max(5, len(y) // 5))
        return tr, y - tr
    raise ValueError(f"Unknown detrend method: {method}")


def period_scan(x: np.ndarray, residual: np.ndarray, cfg: Config, rng: np.random.Generator) -> Dict[str, float]:
    x = np.asarray(x, float)
    r = np.asarray(residual, float)
    r = r - np.mean(r)
    if len(x) < cfg.min_points_per_window or float(np.std(r)) <= 1e-14:
        return dict(best_period=np.nan, best_omega=np.nan, best_power=0.0, fap=1.0, amp_std=0.0, phase=np.nan)
    periods = np.linspace(cfg.period_min_Z, cfg.period_max_Z, cfg.n_periods_Z)
    ss = float(np.sum(r * r))
    powers = []
    coeffs = []
    for p in periods:
        w = 2 * np.pi / p
        X = np.column_stack([np.cos(w * x), np.sin(w * x), np.ones_like(x)])
        beta, *_ = np.linalg.lstsq(X, r, rcond=None)
        fit = X @ beta
        power = float(np.sum((fit - np.mean(fit)) ** 2) / max(ss, 1e-30))
        powers.append(max(0.0, min(1.0, power)))
        coeffs.append(beta)
    powers = np.array(powers)
    j = int(np.argmax(powers))
    null_max = []
    for _ in range(cfg.n_permutations):
        rp = rng.permutation(r)
        ssn = float(np.sum((rp - np.mean(rp)) ** 2))
        mx = 0.0
        for p in periods:
            w = 2 * np.pi / p
            X = np.column_stack([np.cos(w * x), np.sin(w * x), np.ones_like(x)])
            beta, *_ = np.linalg.lstsq(X, rp, rcond=None)
            fit = X @ beta
            powv = float(np.sum((fit - np.mean(fit)) ** 2) / max(ssn, 1e-30))
            mx = max(mx, powv)
        null_max.append(mx)
    fap = (1.0 + float(np.sum(np.array(null_max) >= powers[j]))) / (cfg.n_permutations + 1.0)
    beta = coeffs[j]
    amp = float(math.hypot(float(beta[0]), float(beta[1])) / max(float(np.std(r)), 1e-30))
    phase = float(math.atan2(-float(beta[1]), float(beta[0])))
    return dict(
        best_period=float(periods[j]),
        best_omega=float(2 * np.pi / periods[j]),
        best_power=float(powers[j]),
        fap=float(fap),
        amp_std=float(amp),
        phase=phase,
        null_max_power_median=float(np.median(null_max)),
    )


def cv(values: Iterable[float]) -> float:
    vals = np.array([v for v in values if np.isfinite(v)], dtype=float)
    if len(vals) <= 1 or float(np.mean(vals)) == 0:
        return float("inf")
    return float(np.std(vals) / abs(np.mean(vals)))


def ks_distance(a: List[int], b: List[int]) -> float:
    if len(a) == 0 or len(b) == 0:
        return float("nan")
    aa = np.sort(np.asarray(a, dtype=float))
    bb = np.sort(np.asarray(b, dtype=float))
    grid = np.unique(np.concatenate([aa, bb]))
    fa = np.searchsorted(aa, grid, side="right") / len(aa)
    fb = np.searchsorted(bb, grid, side="right") / len(bb)
    return float(np.max(np.abs(fa - fb)))


def build_cases(cfg: Config, Z: np.ndarray) -> Dict[str, Dict[int, Dict[str, np.ndarray]]]:
    rng = np.random.default_rng(cfg.random_seed)
    cases: Dict[str, Dict[int, Dict[str, np.ndarray]]] = {
        "H1": {},
        "CTRL_markov_stationary_no_Z": {},
        "CTRL_Bernoulli_independent_lattice": {},
        "CTRL_nonlattice_matched_activation": {},
        "CTRL_Z_design_randomized": {},
        "CTRL_phase_scrambled_by_level": {},
        "CTRL_semiMarkov_matched_dwell": {},
        "CTRL_matched_heatkernel_envelope": {},
        "POS_logperiodic_detector_check": {},
    }

    for i in cfg.levels:
        # H1 no explicit log-periodic injection.
        p_h1 = activation_probability(Z, cfg, i, "H1")
        chi_h1 = simulate_markov_from_p(p_h1, cfg, rng, independent=False)
        act_h1 = chi_h1.mean(axis=0)
        cases["H1"][i] = {"p": p_h1, "chi": chi_h1, "activation": act_h1}

        # Controls.
        p_stat = activation_probability(Z, cfg, i, "CTRL_markov_stationary_no_Z")
        chi_stat = simulate_markov_from_p(p_stat, cfg, rng, independent=False)
        cases["CTRL_markov_stationary_no_Z"][i] = {"p": p_stat, "chi": chi_stat, "activation": chi_stat.mean(axis=0)}

        chi_bern = simulate_markov_from_p(p_h1, cfg, rng, independent=True)
        cases["CTRL_Bernoulli_independent_lattice"][i] = {"p": p_h1, "chi": chi_bern, "activation": chi_bern.mean(axis=0)}

        p_nonlat = activation_probability(Z, cfg, i, "CTRL_nonlattice_matched_activation")
        chi_nonlat = simulate_markov_from_p(p_nonlat, cfg, rng, independent=False)
        cases["CTRL_nonlattice_matched_activation"][i] = {"p": p_nonlat, "chi": chi_nonlat, "activation": chi_nonlat.mean(axis=0)}

        chi_zrand = shuffled_by_level(chi_h1, rng)
        cases["CTRL_Z_design_randomized"][i] = {"p": np.full_like(Z, np.mean(p_h1)), "chi": chi_zrand, "activation": chi_zrand.mean(axis=0)}

        act_phase = np.clip(phase_scramble_curve(act_h1, rng), 0, 1)
        # Bernoulli sampling around phase-scrambled activation to have runs.
        chi_phase = simulate_markov_from_p(np.clip(act_phase, 1e-3, 1 - 1e-3), cfg, rng, independent=False)
        cases["CTRL_phase_scrambled_by_level"][i] = {"p": act_phase, "chi": chi_phase, "activation": chi_phase.mean(axis=0)}

        h1_runs = run_lengths(chi_h1)
        chi_semi = reconstruct_from_run_lengths(h1_runs, cfg.n_realizations, len(Z), float(np.mean(act_h1)), rng)
        cases["CTRL_semiMarkov_matched_dwell"][i] = {"p": np.full_like(Z, np.mean(act_h1)), "chi": chi_semi, "activation": chi_semi.mean(axis=0)}

        p_env = activation_probability(Z, cfg, i, "CTRL_matched_heatkernel_envelope")
        cases["CTRL_matched_heatkernel_envelope"][i] = {"p": p_env, "chi": np.tile((p_env > 0.5).astype(np.int8), (cfg.n_realizations, 1)), "activation": p_env}

        p_pos = activation_probability(Z, cfg, i, "POS_logperiodic_detector_check")
        chi_pos = simulate_markov_from_p(p_pos, cfg, rng, independent=False)
        cases["POS_logperiodic_detector_check"][i] = {"p": p_pos, "chi": chi_pos, "activation": chi_pos.mean(axis=0)}

    # Derive observables for each case/level.
    for case, per_level in cases.items():
        for i, d in per_level.items():
            act = d["activation"]
            d["spectral_contrast"] = spectral_contrast_from_activation(act, i, cfg)
            d["heat_proxy"] = heat_contrast_from_activation(act, i, cfg)
    return cases


def audit_window(cfg: Config, Z: np.ndarray, cases: Dict[str, Dict[int, Dict[str, np.ndarray]]], w0: float, w1: float) -> Tuple[List[dict], List[dict], List[dict]]:
    rng = np.random.default_rng(cfg.random_seed + int((w0 + 5) * 1000) + int((w1 + 5) * 100))
    detrenders = ("poly2", "poly3", "moving")
    observables = ("activation", "spectral_contrast", "heat_proxy")
    level_rows: List[dict] = []
    case_rows: List[dict] = []
    dwell_rows: List[dict] = []

    # First compute level/observable/detrender metrics.
    for case, per_level in cases.items():
        for i in cfg.levels:
            zc = level_center(cfg, i, nonlattice=False)
            zeta = Z - zc
            mask = (zeta >= w0) & (zeta <= w1)
            if int(np.sum(mask)) < cfg.min_points_per_window:
                continue
            x = zeta[mask]
            d = per_level[i]
            act = d["activation"][mask]
            chi_win = d["chi"][:, mask]
            runs_on = run_lengths(chi_win)
            dwell_rows.append({


Overwriting /mnt/data/ROSG_Test_H1_MesoscopicDSI_Discriminant_Audit.py
